# EDA 11 (refined eda 7) — Alluvials for case studies, aligned to the EDA 9 mechanism tree

**What changed vs the 20260704 version:**

1. **Harrow is no longer the exodus exemplar.** 
   EDA 9 showed Harrow 008/029's poorer-outflow arm is majority *within-London* (ext arm share 0.43/0.41): 
   - from national D8, 
   - nearly all of London counts as poorer, 
   so their cascade reads as the **national-ladder artefact**, not literal departure. 
   Figure C is retitled accordingly (panels unchanged).
2. **New Figure D — the literal exodus.** 
   Croydon 044 and Kingston upon Thames 019:
   - inflow shares 0.03–0.04 (matching Harrow's), 
   - but ext arm shares 0.72/0.69 
      — the majority of the poorer-outflow arm crosses the London boundary. 
   - Same near-zero inflow, opposite destination: the pair isolates exactly what separates artefact from exodus.
3. **Stats footer extended** with `out-arm ext` — the external share of the poorer-outflow
   arm, the EDA 9 leaf-splitting statistic (≥ 0.5 → external-majority).

In every panel:
- **Left** = inflows into the MSOA; **right** = outflows. Band height ∝ flow size (shared scale within a figure).
- **Red family** = cascade arms (from-wealthier in, to-poorer out); **purple family** = counter arms.
- **Pale shades** cross the London boundary (the grey *rest of England* margins); **solid shades** stay within London.
- Footer: `dominance(nat)` · `inflow share` (cascade-churn share on the inflow arm) ·
  **`out-arm ext`** (external share of the Outflow_Poorer arm — EDA 9 split at 0.5) ·
  `external(cross-decile)` (external share of all cross-decile churn) · `dominance(London)` (Frame A).

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.path import Path
from matplotlib.patches import PathPatch

plt.rcParams['font.family'] = 'DejaVu Sans'
from pyprojroot import here

In [ ]:
ROOT       = here()
sys.path.insert(0, str(ROOT))

DATA_DIR   = ROOT / 'data'
OUT_DIR    = ROOT / 'outputs' / 'case_study'
NAT  = ROOT / 'outputs'/'msoa_cascade_national_frame_20260625.csv'
EDA4 = ROOT / 'outputs'/'eda4_results_for_phase3_20260626.csv'
EXT_DECILE = 6                                   # synthetic external node sits at national D6
YL = 3.05

OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
C_IN, C_EX = '#c0392b', '#e8a7a0'
K_IN, K_EX = '#6a51a3', '#bcaede'
BAR  = '#2f2f2f'
ZONE = '#f2f0ec'          # rest-of-England margin shading
BND  = '#8a8a8a'          # boundary dash colour

In [ ]:
# ── x-geometry ──────────────────────────────────────────────────────────
X_EXT_L = -1.05   # external ribbons start here (left)
X_BND_L =  0.30   # London boundary, left
X_ML    =  3.30   # MSOA bar left edge
X_MR    =  3.90   # MSOA bar right edge
X_BND_R =  6.90   # London boundary, right
X_EXT_R =  8.25   # external ribbons end here (right)

INNER_LONDON = {
    'Camden','City of London','Greenwich','Hackney','Hammersmith and Fulham',
    'Haringey','Islington','Kensington and Chelsea','Lambeth','Lewisham',
    'Newham','Southwark','Tower Hamlets','Wandsworth','Westminster',
}

In [ ]:
def _band(ax, x0, x1, lo, hi, color):
    ax.add_patch(mpatches.Rectangle((x0, lo), x1-x0, hi-lo,
                                    facecolor=color, edgecolor='none',
                                    alpha=0.92, zorder=3))

In [ ]:
def _chevron(ax, xc, yc, h):
    """White right-pointing chevron centred at (xc, yc); h = band height."""
    s = min(h*0.55, 0.14)
    if s < 0.045: return
    w = s * 0.9
    verts = [(xc-w/2, yc+s), (xc+w/2, yc), (xc-w/2, yc-s),
             (xc-w/2+w*0.45, yc), (xc-w/2, yc+s)]
    ax.add_patch(PathPatch(Path(verts, closed=True), facecolor='white',
                           edgecolor='none', alpha=0.85, zorder=4))

In [ ]:
def _lay(segs, gap, scale):
    hs = [(v/scale, c, n, x0, x1) for v, c, n, x0, x1 in segs]
    tot = sum(h for h, *_ in hs) + gap*(sum(1 for _, _, n, _, _ in hs if n) - 1)
    top = tot/2; out = []
    for i, (h, c, n, x0, x1) in enumerate(hs):
        if n and i > 0: top -= gap
        out.append((top-h, top, c, x0, x1)); top -= h
    return out

In [ ]:
def arms_split(r, yr):
    d  = r['Wealth_Decile_National']
    ei, eo = r[f'Ext_Inflow_nat_{yr}'], r[f'Ext_Outflow_nat_{yr}']
    iw, op = r[f'Inflow_Wealthier_nat_{yr}'], r[f'Outflow_Poorer_nat_{yr}']
    ip, ow = r[f'Inflow_Poorer_nat_{yr}'],    r[f'Outflow_Wealthier_nat_{yr}']
    e_iw = ei if d < EXT_DECILE else 0; e_ip = ei if d > EXT_DECILE else 0
    e_ow = eo if d < EXT_DECILE else 0; e_op = eo if d > EXT_DECILE else 0
    return dict(iw=iw, op=op, ip=ip, ow=ow, iw_e=e_iw, op_e=e_op, ip_e=e_ip, ow_e=e_ow,
                iw_i=iw-e_iw, op_i=op-e_op, ip_i=ip-e_ip, ow_i=ow-e_ow)

In [ ]:
def panel(ax, r, yr, scale, title, dom_lon, show_headers=True):
    a = arms_split(r, yr); gap = 0.30

    # background: rest-of-England margins + boundary dashes
    ax.add_patch(mpatches.Rectangle((X_EXT_L-0.55, -YL), X_BND_L-(X_EXT_L-0.55), 2*YL,
                                    facecolor=ZONE, edgecolor='none', zorder=0))
    ax.add_patch(mpatches.Rectangle((X_BND_R, -YL), (X_EXT_R+0.55)-X_BND_R, 2*YL,
                                    facecolor=ZONE, edgecolor='none', zorder=0))
    for xb in (X_BND_L, X_BND_R):
        ax.plot([xb, xb], [-YL*0.88, YL*0.80], ls=(0,(4,3)), lw=0.9, color=BND, zorder=1)
    ax.text(X_BND_L, -YL*0.96, 'London boundary', ha='center', va='top',
            fontsize=5.8, color=BND)
    ax.text(X_BND_R, -YL*0.96, 'London boundary', ha='center', va='top',
            fontsize=5.8, color=BND)
    ax.text((X_EXT_L-0.55+X_BND_L)/2, -YL*0.80, 'rest of\nEngland', ha='center',
            va='center', fontsize=5.8, color='#9a9a9a', style='italic')
    ax.text((X_BND_R+X_EXT_R+0.55)/2, -YL*0.80, 'rest of\nEngland', ha='center',
            va='center', fontsize=5.8, color='#9a9a9a', style='italic')

    # segments carry their own x-extent: internal stops at the boundary, external crosses it
    Lsegs = [(a['iw_i'], C_IN, True,  X_BND_L, X_ML),
             (a['iw_e'], C_EX, False, X_EXT_L, X_ML),
             (a['ip_i'], K_IN, True,  X_BND_L, X_ML),
             (a['ip_e'], K_EX, False, X_EXT_L, X_ML)]
    Rsegs = [(a['ow_i'], K_IN, True,  X_MR, X_BND_R),
             (a['ow_e'], K_EX, False, X_MR, X_EXT_R),
             (a['op_i'], C_IN, True,  X_MR, X_BND_R),
             (a['op_e'], C_EX, False, X_MR, X_EXT_R)]
    L, R = _lay(Lsegs, gap, scale), _lay(Rsegs, gap, scale)

    for lo, hi, c, x0, x1 in L:
        _band(ax, x0, x1, lo, hi, c)
        _chevron(ax, X_ML-0.55, (lo+hi)/2, hi-lo)          # arrows into the bar
    for lo, hi, c, x0, x1 in R:
        _band(ax, x0, x1, lo, hi, c)
        _chevron(ax, x1-0.45, (lo+hi)/2, hi-lo)            # arrows exiting

    bar_lo, bar_hi = min(L[-1][0], R[-1][0]), max(L[0][1], R[0][1])
    ax.add_patch(mpatches.Rectangle((X_ML, bar_lo), X_MR-X_ML, bar_hi-bar_lo,
                                    facecolor=BAR, edgecolor='none', zorder=5))

    # arm labels + counts, anchored at the OUTER end of each arm
    iw_c=(L[0][0]+L[1][1])/2; ip_c=(L[2][0]+L[3][1])/2
    ow_c=(R[0][0]+R[1][1])/2; op_c=(R[2][0]+R[3][1])/2
    def lab(x, y, name, tot, ext, c, ha):
        if tot/scale < 0.03: return
        ax.text(x, y+0.085, name, ha=ha, va='bottom', fontsize=6.4,
                color=c, fontweight='bold')
        ax.text(x, y-0.075, f'{int(tot)}'+(f' (ext {int(ext)})' if ext > 0.5 else ''),
                ha=ha, va='top', fontsize=6.8, color=c)
    lab(X_EXT_L-0.68, iw_c, 'from wealthier', a['iw'], a['iw_e'], C_IN, 'right')
    lab(X_EXT_L-0.68, ip_c, 'from poorer',    a['ip'], a['ip_e'], K_IN, 'right')
    lab(X_EXT_R+0.68, ow_c, 'to wealthier',   a['ow'], a['ow_e'], K_IN, 'left')
    lab(X_EXT_R+0.68, op_c, 'to poorer',      a['op'], a['op_e'], C_IN, 'left')

    # direction headers, drawn on the plot itself
    if show_headers:
        ax.text((X_EXT_L+X_ML)/2, YL*0.90, 'I N F L O W S  \u25b6',
                ha='center', va='center', fontsize=7.6, color='#555',
                fontweight='bold')
        ax.text((X_MR+X_EXT_R)/2, YL*0.90, '\u25b6  O U T F L O W S',
                ha='center', va='center', fontsize=7.6, color='#555',
                fontweight='bold')

    # geography chip: ring + national decile
    ring = 'Inner' if r['ladnm'] in INNER_LONDON else 'Outer'
    ax.text(0.99, 0.985, f"{ring} London \u00b7 nat D{int(r['Wealth_Decile_National'])}",
            transform=ax.transAxes, ha='right', va='top', fontsize=6.6,
            color='#444',
            bbox=dict(boxstyle='round,pad=0.32', fc='white', ec='#cfcfcf', lw=0.7))

    casc, cnt = a['iw']+a['op'], a['ip']+a['ow']; dom = casc/(casc+cnt)
    infl = a['iw']/(a['iw']+a['op']) if (a['iw']+a['op']) else np.nan
    oa   = a['op_e']/a['op'] if a['op'] else np.nan          # EDA 9 leaf split: ext share of poorer-outflow arm
    ext_sh = (a['iw_e']+a['ip_e']+a['ow_e']+a['op_e'])/(casc+cnt) if (casc+cnt) else 0
    ax.text(0.5, -0.045, f'inflow share {infl:.2f}   '
            f'out-arm ext {oa:.2f}   external(cross-decile) {ext_sh:.0%}',
            transform=ax.transAxes, ha='center', va='top', fontsize=7.6,
            color=C_IN if dom >= 0.5 else K_IN, fontweight='bold')
    ax.text(0.5, 1.0, title, transform=ax.transAxes, ha='center', va='bottom',
            fontsize=10, fontweight='bold')
    ax.set_xlim(X_EXT_L-2.55, X_EXT_R+2.55); ax.set_ylim(-YL, YL)
    ax.set_axis_off()

In [ ]:
# def panel(ax, r, yr, scale, title, dom_lon, show_headers=True):
#     a = arms_split(r, yr); gap = 0.30

#     # background: rest-of-England margins + boundary dashes
#     ax.add_patch(mpatches.Rectangle((X_EXT_L-0.55, -YL), X_BND_L-(X_EXT_L-0.55), 2*YL,
#                                     facecolor=ZONE, edgecolor='none', zorder=0))
#     ax.add_patch(mpatches.Rectangle((X_BND_R, -YL), (X_EXT_R+0.55)-X_BND_R, 2*YL,
#                                     facecolor=ZONE, edgecolor='none', zorder=0))
#     for xb in (X_BND_L, X_BND_R):
#         ax.plot([xb, xb], [-YL*0.88, YL*0.80], ls=(0,(4,3)), lw=0.9, color=BND, zorder=1)
#     ax.text(X_BND_L, -YL*0.96, 'London boundary', ha='center', va='top',
#             fontsize=5.8, color=BND)
#     ax.text(X_BND_R, -YL*0.96, 'London boundary', ha='center', va='top',
#             fontsize=5.8, color=BND)
#     ax.text((X_EXT_L-0.55+X_BND_L)/2, -YL*0.80, 'rest of\nEngland', ha='center',
#             va='center', fontsize=5.8, color='#9a9a9a', style='italic')
#     ax.text((X_BND_R+X_EXT_R+0.55)/2, -YL*0.80, 'rest of\nEngland', ha='center',
#             va='center', fontsize=5.8, color='#9a9a9a', style='italic')

#     # segments carry their own x-extent: internal stops at the boundary, external crosses it
#     Lsegs = [(a['iw_i'], C_IN, True,  X_BND_L, X_ML),
#              (a['iw_e'], C_EX, False, X_EXT_L, X_ML),
#              (a['ip_i'], K_IN, True,  X_BND_L, X_ML),
#              (a['ip_e'], K_EX, False, X_EXT_L, X_ML)]
#     Rsegs = [(a['ow_i'], K_IN, True,  X_MR, X_BND_R),
#              (a['ow_e'], K_EX, False, X_MR, X_EXT_R),
#              (a['op_i'], C_IN, True,  X_MR, X_BND_R),
#              (a['op_e'], C_EX, False, X_MR, X_EXT_R)]
#     L, R = _lay(Lsegs, gap, scale), _lay(Rsegs, gap, scale)

#     for lo, hi, c, x0, x1 in L:
#         _band(ax, x0, x1, lo, hi, c)
#         _chevron(ax, X_ML-0.55, (lo+hi)/2, hi-lo)          # arrows into the bar
#     for lo, hi, c, x0, x1 in R:
#         _band(ax, x0, x1, lo, hi, c)
#         _chevron(ax, x1-0.45, (lo+hi)/2, hi-lo)            # arrows exiting

#     bar_lo, bar_hi = min(L[-1][0], R[-1][0]), max(L[0][1], R[0][1])
#     ax.add_patch(mpatches.Rectangle((X_ML, bar_lo), X_MR-X_ML, bar_hi-bar_lo,
#                                     facecolor=BAR, edgecolor='none', zorder=5))

#     # arm labels + counts, anchored at the OUTER end of each arm
#     iw_c=(L[0][0]+L[1][1])/2; ip_c=(L[2][0]+L[3][1])/2
#     ow_c=(R[0][0]+R[1][1])/2; op_c=(R[2][0]+R[3][1])/2
#     def lab(x, y, name, tot, ext, c, ha):
#         if tot/scale < 0.03: return
#         ax.text(x, y+0.085, name, ha=ha, va='bottom', fontsize=6.4,
#                 color=c, fontweight='bold')
#         ax.text(x, y-0.075, f'{int(tot)}'+(f' (ext {int(ext)})' if ext > 0.5 else ''),
#                 ha=ha, va='top', fontsize=6.8, color=c)
#     lab(X_EXT_L-0.68, iw_c, 'from wealthier', a['iw'], a['iw_e'], C_IN, 'right')
#     lab(X_EXT_L-0.68, ip_c, 'from poorer',    a['ip'], a['ip_e'], K_IN, 'right')
#     lab(X_EXT_R+0.68, ow_c, 'to wealthier',   a['ow'], a['ow_e'], K_IN, 'left')
#     lab(X_EXT_R+0.68, op_c, 'to poorer',      a['op'], a['op_e'], C_IN, 'left')

#     # direction headers, drawn on the plot itself
#     if show_headers:
#         ax.text((X_EXT_L+X_ML)/2, YL*0.90, 'I N F L O W S  \u25b6',
#                 ha='center', va='center', fontsize=7.6, color='#555',
#                 fontweight='bold')
#         ax.text((X_MR+X_EXT_R)/2, YL*0.90, '\u25b6  O U T F L O W S',
#                 ha='center', va='center', fontsize=7.6, color='#555',
#                 fontweight='bold')

#     # geography chip: ring + national decile
#     ring = 'Inner' if r['ladnm'] in INNER_LONDON else 'Outer'
#     ax.text(0.99, 0.985, f"{ring} London \u00b7 nat D{int(r['Wealth_Decile_National'])}",
#             transform=ax.transAxes, ha='right', va='top', fontsize=6.6,
#             color='#444',
#             bbox=dict(boxstyle='round,pad=0.32', fc='white', ec='#cfcfcf', lw=0.7))

#     casc, cnt = a['iw']+a['op'], a['ip']+a['ow']; dom = casc/(casc+cnt)
#     infl = a['iw']/(a['iw']+a['op']) if (a['iw']+a['op']) else np.nan
#     oa   = a['op_e']/a['op'] if a['op'] else np.nan          # EDA 9 leaf split: ext share of poorer-outflow arm
#     ext_sh = (a['iw_e']+a['ip_e']+a['ow_e']+a['op_e'])/(casc+cnt) if (casc+cnt) else 0
#     ax.text(0.5, -0.045, f'dominance(nat) {dom:.2f}   inflow share {infl:.2f}   '
#             f'out-arm ext {oa:.2f}   external(cross-decile) {ext_sh:.0%}   \u00b7   dominance(London) {dom_lon:.2f}',
#             transform=ax.transAxes, ha='center', va='top', fontsize=7.6,
#             color=C_IN if dom >= 0.5 else K_IN, fontweight='bold')
#     ax.text(0.5, 1.0, title, transform=ax.transAxes, ha='center', va='bottom',
#             fontsize=10, fontweight='bold')
#     ax.set_xlim(X_EXT_L-2.55, X_EXT_R+2.55); ax.set_ylim(-YL, YL)
#     ax.set_axis_off()

In [ ]:
nat = pd.read_csv(NAT); e4 = pd.read_csv(EDA4)
domA = lambda c, yr: e4.loc[e4['msoa11cd']==c, f'Dom_A_{yr}'].iloc[0]
row  = lambda c: nat[nat['msoa11cd']==c].iloc[0]

In [ ]:
def figscale(cases):
    m = 0
    for c, _ in cases:
        r = row(c)
        for yr in ['11','21']:
            a = arms_split(r, yr); m = max(m, a['iw']+a['ip'], a['ow']+a['op'])
    return m/3.0

LEG = [mpatches.Patch(color=C_IN, label='Cascade \u00b7 within London'),
       mpatches.Patch(color=C_EX, label='Cascade \u00b7 crossing London boundary'),
       mpatches.Patch(color=K_IN, label='Counter \u00b7 within London'),
       mpatches.Patch(color=K_EX, label='Counter \u00b7 crossing London boundary')]

def build(title, cases, fname, caption=''):
    sc = figscale(cases); nr = len(cases)
    fig, axes = plt.subplots(nr, 2, figsize=(14, 3.4*nr+1.4), squeeze=False)
    for i, (c, n) in enumerate(cases):
        panel(axes[i,0], row(c), '11', sc, f'{n}  \u00b7  2011', domA(c,'11'), show_headers=(i==0))
        panel(axes[i,1], row(c), '21', sc, f'{n}  \u00b7  2021', domA(c,'21'), show_headers=(i==0))
    fig.suptitle(title, fontsize=13.5, fontweight='bold', y=0.995)
    fig.legend(handles=LEG, loc='lower center', ncol=4, frameon=False, fontsize=8.4,
               bbox_to_anchor=(0.5, 0.006))
    if caption:
        fig.text(0.5, 0.048, caption, ha='center', va='bottom', fontsize=8.2,
                 style='italic', color='#333', wrap=True)
    plt.tight_layout(rect=[0, 0.07 if caption else 0.035, 1, 0.975])
    plt.savefig(OUT_DIR / fname, dpi=130, bbox_inches='tight'); plt.show()

---
## Figure A — persistent (genuine) cascade *(unchanged; footer stat added)*
Two exemplars of the 8 MSOAs that are inflow-driven in both censuses.

In [ ]:
build('Persistent cascade \u2014 affluent inflow AND poorer outflow, robust across frames',
      [('E02000191','Camden 026'),('E02000873','Tower Hamlets 010')],
      'fig_alluvial_case_A_persistent_cascade.png')

## Figure B — inner cascade turning counter *(unchanged; footer stat added)*

In [ ]:
build('Inner cascade turning counter \u2014 the city-wide 2021 reversal',
      [('E02000809','Southwark 003'),('E02000561','Islington 008'),('E02000957','Wandsworth 035')],
      'fig_alluvial_case_B_cascade_to_counter.png')

## Figure C — Harrow, reframed: the national-ladder artefact *(retitled)*

Same panels as before; the title no longer claims exodus. 
- Inflow shares 0.01–0.04, but
- `out-arm ext` = 0.43/0.41 — the **majority of the poorer outflow stays within London**.

From national D8, 69% of London MSOAs count as poorer, so the cascade reading here is manufactured by the national ladder, not by departure. 

EDA 9 leaf: *outflow, internal-majority*.

In [ ]:
build('Outflow-dominated cascade, internal-majority \u2014 the national-ladder artefact',
      [('E02000440','Harrow 008'),('E02000461','Harrow 029')],
      'fig_alluvial_case_C_ladder_artefact.png')

## Figure D — the literal exodus *(new)*

Croydon 044 and Kingston upon Thames 019: 
- inflow shares 0.03/0.04 — the same bare thread as Harrow 
— but `out-arm ext` = 0.72/0.69: 
    - the **majority of the poorer-outflow arm crosses the London boundary** (thick pale-red ribbons on the right). 
    - Same near-zero affluent inflow,
opposite destination: this pair against Figure C isolates exactly what separates the ladder artefact from the exodus.

The pair also embodies the class's temporal story (19 → 78 MSOAs, ×4.1, 100% outer London):
- **Kingston 019 was external-majority already in 2011** (a member of the 19-strong baseline),
- while **Croydon 044 was not cascade-led at all in 2011** and *emerged* into the exodus class by 2021 

One baseline member, one emergent member, in a single figure.

In [ ]:
build('Exodus \u2014 external-majority outflow: affluent departure across the London boundary',
      [('E02000237','Croydon 044'),('E02000616','Kingston upon Thames 019')],
      'fig_alluvial_case_D_exodus_external.png')

---
### Case roster after the EDA 9 alignment

| figure | MSOAs | EDA 9 leaf (2021) | narrative |
|---|---|---|---|
| A | Camden 026 · Tower Hamlets 010 | inflow-driven (both years) | genuine cascade (2 of 8) |
| B | Southwark 003 · Islington 008 · Wandsworth 035 | counter-led | the decoupling / 2021 reversal |
| C | Harrow 008 · Harrow 029 | outflow, internal-majority | ladder artefact — **not** exodus |
| D | Croydon 044 · Kingston u T 019 | outflow, external-majority | the literal exodus (Kingston: 2011-baseline member; Croydon: emergent by 2021) |

C and D share near-zero inflow shares; only the out-arm destination differs. That contrast is
the visual proof that "exodus" is a measured mechanism, not a relabel of outflow.